In [1]:
import requests
import os

BASE = "http://localhost:8000"
VIDEO_PATH = "/home/lisa/Arupreza/AccentFlow-0.2/storage/Sample_2.mp4"

In [ ]:
r = requests.get(f"{BASE}/health")
print("Status:", r.status_code)
print("Response:", r.json())

Status: 200
Response: {'status': 'ok', 'service': 'orchestrator'}


## Extract Audio (Upload Video → Download Audio)

In [3]:
VIDEO_PATH = "storage/Sample_2.mp4"

with open(VIDEO_PATH, "rb") as f:
    r = requests.post(
        f"{BASE}/process",
        files={"video": f},
        data={"max_iterations": 3},
        timeout=600,
    )

if r.status_code == 200:
    result = r.json()
    print("Transcript:", result.get("transcript"))
    print("Final:", result.get("final_text"))
    print("Score:", result.get("grammar_score"))
    print("Audio (in container):", result.get("audio_path"))
    print("Job ID:", result.get("job_id"))
else:
    print("Error:", r.status_code, r.text)

Transcript:  Hello, I am Emre Rajan Islam from Bangladesh. I am a data science engineer and pursuing my PhD in Sunchung Iyang University. I started my research in 2021 as a Master's Student and my research started with Big Data Analysis on Invehicle Network. What is Invehicle Network? Invehicle Network is inside vehicle called Communication Protocol. In this picture, what we can see inside vehicle, there is a lots of small blocks we can see, which is a small computer, and they need to communication with each other. For that reason, scientists built one communication protocol dedicated for IVN. So, it's dedicated for IVN, which is in-vehicle network, and this is called CAN. After introducing CAN, lots of problems solved but few disadvantages also come out. One of them is in CAN there is no inbuilt security mechanism. For that reason, attacker can easily control the car and hack the car. So in this video we can see, see they are controlling the car remotely and the driver don't have any 

## Voice Clone

In [4]:
import requests, ormsgpack, os
import soundfile as sf
from scipy.signal import resample_poly
import numpy as np
from IPython.display import Audio, display

REFERENCE_AUDIO = "/home/lisa/Arupreza/AccentFlow-0.2/extracted_audio.wav"
OUTPUT_FILE     = "voice_cloned.wav"
RESAMPLED       = "/home/lisa/Arupreza/AccentFlow-0.2/storage/reference_22k.wav"

TARGET_TEXT = result["final_text"]   # corrected English from /process
print("✓ Target text:", TARGET_TEXT[:100])

# STEP 1: resample reference (soundfile + scipy)
wav, sr = sf.read(REFERENCE_AUDIO)
if wav.ndim > 1:
    wav = wav.mean(axis=1)                 # mono
if sr != 22050:
    wav = resample_poly(wav, 22050, sr)    # polyphase resample
    sr = 22050
wav = wav[:22050 * 20]                      # max 20s
sf.write(RESAMPLED, wav, 22050)
print(f"✓ Resampled → {sr} Hz")

# STEP 2: transcript of reference
r = requests.post("http://localhost:8005/transcribe",
                    json={"audio_path": "/app/storage/reference_22k.wav"}, timeout=300)
r.raise_for_status()
reference_text = r.json()["transcript"].strip()
print(f"✓ Reference transcript: {reference_text[:100]}...")

# STEP 3: voice clone
with open(RESAMPLED, "rb") as f:
    ref_audio_bytes = f.read()
payload = {
    "text": TARGET_TEXT,
    "references": [{"audio": ref_audio_bytes, "text": reference_text}],
    "format": "wav", "max_new_tokens": 1024, "chunk_length": 200,
    "top_p": 0.7, "repetition_penalty": 1.2, "temperature": 0.7,
    "streaming": False, "use_memory_cache": "off",
}
r = requests.post("http://localhost:8003/v1/tts",
                    headers={"content-type": "application/msgpack"},
                    data=ormsgpack.packb(payload), timeout=600)
if r.status_code == 200:
    with open(OUTPUT_FILE, "wb") as f:
        f.write(r.content)
    print(f"✓ Voice cloned: {OUTPUT_FILE} ({len(r.content)/1024:.1f} KB)")
    display(Audio(OUTPUT_FILE))
else:
    print("Error:", r.status_code, r.text[:500])

✓ Target text: Hello, I am Emre Rajan Islam from Bangladesh. I am a data science engineer and pursuing my PhD at Su
✓ Resampled → 22050 Hz


✓ Reference transcript: Hello, I am Emre Rajan Islam from Bangladesh. I am a data science engineer and pursuing my PhD in Su...
✓ Voice cloned: voice_cloned.wav (4092.0 KB)


## LipSync

In [ ]:
import requests, json

video_path = "/home/lisa/Arupreza/AccentFlow-0.2/storage/Sample_2.mp4"
audio_path = "/home/lisa/Arupreza/AccentFlow-0.2/voice_cloned.wav"

with open(video_path, "rb") as v, open(audio_path, "rb") as a:
    r = requests.post(
        "http://localhost:8004/sync",
        files={"video": v, "audio": a},
        data={"bbox_shift": 0, "fps": 25, "use_float16": True},
        timeout=None,
    )

print("Status:", r.status_code)
if r.status_code == 200:
    out = r.json()
    print(json.dumps(out, indent=2))   # inspect: find the output path key
else:
    print(r.text[:2000])